In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.volume import OnBalanceVolumeIndicator, ChaikinMoneyFlowIndicator
from ta.volatility import AverageTrueRange
from ta.volume import VolumeWeightedAveragePrice
from scipy.optimize import minimize
from datetime import datetime
from dateutil.relativedelta import relativedelta
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb



from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')



TICKERS = [
        "PETR4.SA", "VALE3.SA", "PRIO3.SA",
        "MGLU3.SA", "LREN3.SA", "ABEV3.SA", "WEGE3.SA",
        "ELET3.SA",
        "SUZB3.SA",
        "EMBR3.SA", "RDOR3.SA", "RAIL3.SA"
    ]


TICKERS_EXPANDIDA = [
    # BANCOS (11)
    "ITUB4.SA",  # Itaú Unibanco
    "BBDC4.SA",  # Bradesco
    "BBAS3.SA",  # Banco do Brasil
    "SANB11.SA", # Santander
    "BPAC11.SA", # Banco do Brasil PN
    "CXSE3.SA",  # Caixa Seguridade
    "BRAP4.SA",  # Bradespar
    "BRSR6.SA",  # Banco do Brasil ON
    "CRFB3.SA",  # Carrefour Brasil
    "PSSA3.SA",  # Porto Seguro
    "PINE4.SA",  # Banco Pine
    
    # ENERGIA (12)
    "PETR4.SA",  # Petrobras PN
    "PRIO3.SA",  # Petrorio
    "OIBR4.SA",  # Oi PN
    "ELET3.SA",  # Eletrobras ON
    "CMIG4.SA",  # Cemig PN
    "CPFE3.SA",  # CPFL Energia
    "EGIE3.SA",  # EDP Energias
    "ENGI11.SA", # Engie Brasil
    "GEMA3.SA",  # Gerdau Metalúrgica
    "LIGHT3.SA", # Light
    "TRPL4.SA",  # Transmissão Paulista
    "EQTL3.SA",  # Equatorial Energia
    
    # MINERAÇÃO (4)
    "VALE3.SA",  # Vale
    "CSNA3.SA",  # Companhia Siderúrgica
    "USIM5.SA",  # Usiminas
    "GGBR4.SA",  # Gerdau PN
    
    # VAREJO (8)
    "MGLU3.SA",  # Magazine Luiza
    "LREN3.SA",  # Lojas Renner
    "ABEV3.SA",  # Ambev
    "RENT3.SA",  # Localiza
    "MOVI3.SA",  # Movida
    "VVAR3.SA",  # Via Varejo
    "PCAR3.SA",  # Impar
    "TRIS3.SA",  # Triscila
    
    # CONSUMO (9)
    "WEGE3.SA",  # WEG
    "JBSS3.SA",  # JBS
    "MSFT34.SA", # Microsoft (ADR)
    "HYPE3.SA",  # Hypera
    "SLCE3.SA",  # SLC Agrícola
    "PETZ3.SA",  # Petz
    "ARZZ3.SA",  # Arezzo
    "TFCO4.SA",  # Telefônico Brasil
    "BRML3.SA",  # Brasil Malha Logística
    
    # TRANSPORTE (6)
    "RAIL3.SA",  # Rumo
    "CCRO3.SA",  # CCR
    "LOGB3.SA",  # Loggi
    "ARZZ3.SA",  # Arezzo (calçados)
    "EMAE4.SA",  # Emae
    "ATUS3.SA",  # Atus
    
    # CONSTRUÇÃO (5)
    "MRVE3.SA",  # MRV Engenharia
    "TEND3.SA",  # Construtora Tenda
    "PLPL3.SA",  # Plano & Plano
    "GFSA3.SA",  # Gafisa
    "TRAD3.SA",  # Tradição
    
    # IMÓVEIS (5)
    "VLID3.SA",  # Validada Imóveis
    "BRIV3.SA",  # BR Imobiliário
    "CYRE3.SA",  # Cyrela
    "EVEN3.SA",  # Even
    "HBOR3.SA",  # Helbor
    
    # COMUNICAÇÃO (3)
    "VIVT3.SA",  # Vivo
    "TIMS3.SA",  # Tim
    "OIBR3.SA",  # Oi ON
    
    # PAPEL E CELULOSE (4)
    "SUZB3.SA",  # Suzano
    "SBSP3.SA",  # Sabesp
    "KLABIN11.SA", # Klabin
    "FIBR3.SA",  # Fibria
    
    # QUÍMICA/HIGIENE (3)
    "TOTS3.SA",  # Totvs
    "BRPR3.SA",  # Brasilfops
    "CLSA3.SA",  # Classa
    
    # ALIMENTOS (4)
    "MBLY3.SA",  # Marfrig
    "BRF3.SA",   # BRF
    "SEQL3.SA",  # Sequoia
    "ASAI3.SA",  # Assaí
    
    # TECNOLOGIA (5)
    "TOTS3.SA",  # Totvs
    "NTCO3.SA",  # Natura
    "BRQT3.SA",  # Brq Digital
    "DIRR3.SA",  # Direcional Engenharia
    "TRPL4.SA",  # Transmissão Paulista
    
    # AVIAÇÃO (3)
    "EMBR3.SA",  # Embraer
    "AZUL4.SA",  # Azul
    "GOLL4.SA",  # Gol
    
    # SEGUROS (3)
    "PSSA3.SA",  # Porto Seguro
    "SULB3.SA",  # Sulamerica
    "SGUP3.SA",  # Seguradoras Unidas
    
    # FINANCEIRAS (4)
    "B3SA3.SA",  # B3
    "MOVI3.SA",  # Movida
    "RBRR3.SA",  # Rede Brasil Real
    "RDOR3.SA",  # Rede D'Or
    
    # AGRONEGÓCIO (3)
    "AGRO3.SA",  # Agrogalaxy
        "AERI3.SA",  # Aerea Invest
    "POSI3.SA",  # Positivo
    ]

In [2]:
import pandas as pd
from ta.trend import (
    SMAIndicator, EMAIndicator, WMAIndicator, MACD
)
from ta.momentum import (
    RSIIndicator, StochasticOscillator, ROCIndicator
)
from ta.volatility import (
    AverageTrueRange, BollingerBands, KeltnerChannel
)
from ta.volume import OnBalanceVolumeIndicator

In [3]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint
import numpy as np
import pandas as pd

In [4]:
import pandas as pd
import yfinance as yf
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange

In [5]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from typing import Dict, Any, Union
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

In [6]:
from statsmodels import api as sm

In [1]:
def preparar_features_target(df, target_lag=1):
    """
    Prepara features defasadas e target SEM VAZAMENTO DE DADOS.
    
    CORREÇÃO CRÍTICA:
    - Calcula o target ANTES de fazer qualquer shift nas features
    - O target representa o retorno observado 21 dias à frente
    - As features são defasadas em 1 período para não usar info do dia da predição
    - Ao treinar, a data de rebalanceamento não pode ter observado esse target
    """
    data = df.copy()
    
    # 1. PRIMEIRO: Calcular o target futuro (antes de qualquer shift)
    data['Close_Future'] = data['Close'].shift(-target_lag)
    data['Target'] = (data['Close_Future'] - data['Close']) / data['Close']
    
    # 2. DEPOIS: Defasar as features em 1 período
    feature_cols = ['SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'EMA_50',
                    'RSI_14', 'MACD', 'MACD_Hist', 'ATR_14', 'ROC_12', 'OBV',
                    'Ret_1d', 'Ret_5d', 'Ret_21d', 'Vol_21d', 'Vol_63d']
    
    for col in feature_cols:
        data[f'{col}_lag1'] = data[col].shift(1)
    
    # 3. REMOVER últimos target_lag linhas (pois não têm target válido)
    data = data.iloc[:-target_lag].copy()
    
    # 4. Usar apenas features defasadas
    feature_cols_lag = [col for col in data.columns if col.endswith('_lag1')]
    data = data.dropna(subset=['Target'] + feature_cols_lag).copy()
    
    return data, feature_cols_lag


def treinar_e_prever(df, feature_cols_lag, data_predicao, ticker, modelo_tipo='RF', target_lag=1):
    """
    Treina modelo SEM VAZAMENTO DE DADOS - CORRIGIDO COM LÓGICA DE PREDIÇÃO.
    
    LÓGICA CRÍTICA:
    - Se queremos PREVER em data_predicao, e temos lag de 21 dias
    - O target de cada linha = retorno 21 dias no FUTURO
    - Portanto, para treinar SEM VAZAMENTO:
      * Treinar APENAS até (data_predicao - target_lag)
      * Isso garante que o target da última linha será observado apenas APÓS data_predicao
    
    - PARA FAZER A PREDIÇÃO:
      * Queremos features do dia data_predicao (ou próximo dia útil)
      * Mas essas features NÃO FORAM CALCULADAS NOS DADOS DE TREINO
      * Precisamos buscar NO DATAFRAME ORIGINAL as features para data_predicao
      * E passar para o modelo treinado
    
    Exemplo:
    - Queremos prever retorno em 31/08/2025
    - Treinar até: 31/08 - 21 = 10/08 (última linha com target conhecido)
    - Predição: usar features de 31/08 (primeira data >= data_predicao)
    - Resultado: previsão para próximos 21 dias (até 21/09)
    """
    # Data limite de treino: (data_predicao - target_lag)
    data_limite_treino = data_predicao - pd.Timedelta(days=target_lag)
    
    # Treinar APENAS com dados ANTES dessa data
    df_treino = df[df.index <= data_limite_treino].copy()
    
    if len(df_treino) < 252:
        print(f" ❌ Dados insuficientes: {len(df_treino)} dias")
        return None
    
    # Extrair X e y para treino
    X_train = df_treino[feature_cols_lag]
    y_train = df_treino['Target']
    
    # Escolher modelo
    if modelo_tipo == 'RF':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    else:  # OLS
        model = LinearRegression()
    
    model.fit(X_train, y_train)
    
    # ✅ CORREÇÃO: Buscar features para data_predicao no DF ORIGINAL (não no de treino)
    # Encontrar primeira data >= data_predicao
    df_futuro = df[df.index >= data_predicao]
    
    if len(df_futuro) == 0:
        print(f" ❌ Sem dados disponíveis para predição em/após {data_predicao.strftime('%Y-%m-%d')}")
        return None
    
    linha_predicao = df_futuro.iloc[[0]]  # Primeira linha >= data_predicao
    data_real_predicao = linha_predicao.index[0]
    
    # Verificar se features têm NaN
    features_com_nan = linha_predicao[feature_cols_lag].isna().sum().sum()
    if features_com_nan > 0:
        print(f" ⚠ AVISO: {features_com_nan} features com NaN na linha de predição")
    
    # Fazer predição com linha de data_predicao
    X_pred = linha_predicao[feature_cols_lag]
    predicao = model.predict(X_pred)[0]
    
    if predicao > 0:
        print(f"\n{'─'*70}")
        print(f"🔍 MODELO: {modelo_tipo} - {ticker}")
        print(f"📊 DADOS DE TREINO:")
        print(f" • Início: {df_treino.index[0].strftime('%Y-%m-%d')}")
        print(f" • Fim: {df_treino.index[-1].strftime('%Y-%m-%d')} ← ÚLTIMA LINHA COM TARGET CONHECIDO")
        print(f"📊 DADOS DE PREDIÇÃO:")
        print(f" • Data limite treino: {data_limite_treino.strftime('%Y-%m-%d')} (= data_predicao - {target_lag}d)")
        print(f" • Data solicitada: {data_predicao.strftime('%Y-%m-%d')}")
        print(f" • Data real usada: {data_real_predicao.strftime('%Y-%m-%d')} ← FEATURES PARA PREDIÇÃO")
        print(f" • Total dias treino: {len(df_treino)} dias")
        print(f" • Preço em {data_real_predicao.strftime('%Y-%m-%d')}: R$ {linha_predicao['Close'].iloc[0]:.2f}")
        print(f" ✅ Predição: {predicao*100:+.2f}% (para próximos {target_lag} dias)")
    
    return predicao


def calcular_matriz_covariancia(tickers_list, dados_acoes, data_ref, janela=63, target_lag=1):
    """
    Calcula matriz de covariância APENAS com dados históricos.
    
    CORREÇÃO: data_ref deve ser ajustada para (data_ref - target_lag)
    para evitar vazamento temporal
    """
    retornos_hist = []
    
    # Ajustar data de referência: usar dados até (data_ref - target_lag)
    data_limite = data_ref - pd.Timedelta(days=target_lag)
    
    print(f"\nDatas da matriz de covariancia (ajustada para lag={target_lag})")
    print(f"Data referência: {data_ref.strftime('%Y-%m-%d')} → Data limite: {data_limite.strftime('%Y-%m-%d')}")
    
    for ticker in tickers_list:
        df = dados_acoes[ticker]
        # Usar dados até data_limite (não data_ref)
        df_periodo = df[df.index <= data_limite].tail(janela)
        print(f"• {ticker}: {df_periodo.index[0].strftime('%Y-%m-%d')} a {df_periodo.index[-1].strftime('%Y-%m-%d')}")
        
        if len(df_periodo) >= 21:
            retornos = df_periodo['Close'].pct_change().dropna()
            retornos_hist.append(retornos.values)
        else:
            retornos_hist.append(np.zeros(janela-1))
    
    min_len = min(len(r) for r in retornos_hist)
    retornos_df = pd.DataFrame({ticker: r[-min_len:] for ticker, r in zip(tickers_list, retornos_hist)})
    cov_matrix = retornos_df.cov()
    
    return cov_matrix


def otimizar_markowitz(retornos_esperados, cov_matrix, risk_free_rate=0.10/252):
    """
    Otimização de Markowitz: maximizar Sharpe Ratio.
    """
    tickers = list(retornos_esperados.keys())
    n_ativos = len(tickers)
    mu = np.array([retornos_esperados[t] for t in tickers])
    cov = cov_matrix.loc[tickers, tickers].values
    
    def negative_sharpe(weights):
        portfolio_return = np.dot(weights, mu)
        portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
        sharpe = (portfolio_return - risk_free_rate) / portfolio_vol if portfolio_vol > 0 else 0
        return -sharpe
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = tuple((0, 0.4) for _ in range(n_ativos))
    w0 = np.array([1/n_ativos] * n_ativos)
    
    result = minimize(
        negative_sharpe,
        w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000}
    )
    
    if result.success:
        pesos = {ticker: peso for ticker, peso in zip(tickers, result.x) if peso > 0.001}
    else:
        pesos = {ticker: 1/n_ativos for ticker in tickers}
    
    return pesos

def calcular_max_drawdown(retornos):
    """Calcula o máximo drawdown."""
    valor_acumulado = (1 + pd.Series(retornos)).cumprod()
    max_anterior = valor_acumulado.cummax()
    drawdown = (valor_acumulado - max_anterior) / max_anterior
    return drawdown.min()
    

In [18]:
def baixar_e_calcular_indicadores(ticker, start="2010-01-01",target=1, end=None):
    """
    Baixa dados e calcula indicadores técnicos.
    """
    try:
        if end is not None:
            data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        else:
            data = yf.download(ticker, start=start, progress=False, auto_adjust=True)
        
        if data.empty:
            return None
        
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        
        serie = data['Close']

        #for i in range(10,30,5):
        #    for j in range(20,60,5):
        #        if i != j:
        #            data[f'SMA_{i}-{j}'] = SMAIndicator(serie, window=i).sma_indicator() - SMAIndicator(serie, window=j).sma_indicator()
        data['SMA_21'] = SMAIndicator(serie, window=21).sma_indicator()
        #data['SMA_50'] = SMAIndicator(serie, window=50).sma_indicator()
        #data['SMA_5'] = SMAIndicator(serie, window=5).sma_indicator()
        #data['SMA_Change_21-5'] = SMAIndicator(serie, window=5).sma_indicator() - data['SMA_21']
        data[f'SMA_10-20'] = SMAIndicator(serie, window=10).sma_indicator() - SMAIndicator(serie, window=20).sma_indicator()
        data[f'SMA_15-20'] = SMAIndicator(serie, window=15).sma_indicator() - SMAIndicator(serie, window=20).sma_indicator()
        data['SMA_25-30'] = SMAIndicator(serie, window=30).sma_indicator() - SMAIndicator(serie, window=25).sma_indicator()

        data['SMA_25-50'] = SMAIndicator(serie, window=50).sma_indicator() - SMAIndicator(serie, window=25).sma_indicator()

        #data['EMA_12'] = EMAIndicator(serie, window=12).ema_indicator()
        #data['EMA_26'] = EMAIndicator(serie, window=26).ema_indicator()
        #data['EMA_50'] = EMAIndicator(serie, window=50).ema_indicator()
        data['RSI_14'] = RSIIndicator(serie, window=14).rsi()
        
        macd = MACD(serie)
        #data['MACD'] = macd.macd()
        data['MACD_Hist'] = macd.macd_diff()
        #data['ATR_14'] = AverageTrueRange(data['High'], data['Low'], serie, window=14).average_true_range()
        #data['ROC_12'] = ROCIndicator(serie, window=12).roc()
        #data['OBV'] = OnBalanceVolumeIndicator(serie, data['Volume']).on_balance_volume()
        
        # Retornos e Volatilidade
        #data['Ret_1d'] = serie.pct_change(1)
        #data['Ret_5d'] = serie.pct_change(5)
        #data['Ret_21d'] = serie.pct_change(21)
        data['Vol_21d'] = serie.pct_change().rolling(21).std()
        #data['Vol_63d'] = serie.pct_change().rolling(63).std()
        #data['Ticker'] = ticker

        data['TARGET'] = data['Close'].pct_change(periods=-target).shift(-target)
        data['TARGET_Ret'] = data['TARGET'].pct_change()
        #limiar_compra_mensal = 0.005
        #data['TARGET_Sinal'] = (data['TARGET_Ret'] > limiar_compra_mensal).astype(int)
        
        return data.fillna(method='ffill').dropna()
    except Exception as e:
        print(f"Erro ao baixar {ticker}: {e}")
        return None

Infuncional

In [9]:
def baixar_dados_macro(start="2010-01-01", end=None):
    """
    Baixa dados de indicadores macro/globais (IBOV, Ouro, S&P 500, Nasdaq, Cobre, VIX) 
    e calcula retornos e indicadores técnicos.
    """
    
    # NOVOS Mapeamentos dos Tickers
    macro_tickers = ['^BVSP', 'GC=F', '^GSPC', '^IXIC', 'HG=F', '^VIX'] 
    
    if end is not None:
        data_macro = yf.download(macro_tickers, start=start, end=end, progress=False, auto_adjust=True)
    else:
        data_macro = yf.download(macro_tickers, start=start, progress=False, auto_adjust=True)
        
    if data_macro.empty:
        return None

    # Renomeando as colunas para evitar o MultiIndex e padronizar
    # Exemplo: Close_^BVSP -> Close_IBOV
    rename_map = {
        '^BVSP': 'IBOV', 'GC=F': 'OURO', '^GSPC': 'SPX', 
        '^IXIC': 'NDX', 'HG=F': 'COBRE', '^VIX': 'VIX'
    }
    
    data_macro.columns = [f'{col[0]}_{rename_map.get(col[1], col[1])}' for col in data_macro.columns]
    
    # Lista de ativos para processar
    ativos = ['IBOV', 'OURO', 'SPX', 'NDX', 'COBRE', 'VIX']
    features_macro = []
    
    # -----------------------------------------------
    ## 📈 Loop de Cálculo de Indicadores Técnicos
    # -----------------------------------------------
    
    for ativo in ativos:
        # Pega as séries necessárias para o ativo
        ativo_close = data_macro[f'Close_{ativo}']
        
        # VIX e SPX não têm dados de High/Low/Volume confiáveis/úteis em todas as plataformas
        # Usamos High/Low apenas se existirem. O VIX, por exemplo, não tem volume.
        try:
            ativo_high = data_macro[f'High_{ativo}']
            ativo_low = data_macro[f'Low_{ativo}']
            has_ohl = True
        except KeyError:
            ativo_high = ativo_close
            ativo_low = ativo_close
            has_ohl = False
        
        prefix = f'{ativo}_'
        
        # Retorno e Volatilidade (21 dias)
        data_macro[f'{prefix}Ret_21d'] = ativo_close.pct_change(21)
        data_macro[f'{prefix}Vol_21d'] = ativo_close.pct_change().rolling(21).std()
        
        # Tendência
        data_macro[f'{prefix}SMA_50'] = SMAIndicator(ativo_close, window=50).sma_indicator()
        data_macro[f'{prefix}EMA_200'] = EMAIndicator(ativo_close, window=200).ema_indicator()
        
        # Posição Relativa à Média (Feature útil)
        data_macro[f'{prefix}vs_SMA50'] = ativo_close / data_macro[f'{prefix}SMA_50'] - 1 

        # Momentum (RSI e MACD)
        data_macro[f'{prefix}RSI_14'] = RSIIndicator(ativo_close, window=14).rsi()
        macd_ativo = MACD(ativo_close)
        data_macro[f'{prefix}MACD_Hist'] = macd_ativo.macd_diff() 

        # Volatilidade (ATR) - Requer High e Low
        if has_ohl:
             data_macro[f'{prefix}ATR_14'] = AverageTrueRange(ativo_high, ativo_low, ativo_close, window=14).average_true_range()
        else:
             # Para VIX, onde High/Low pode ser igual a Close
             data_macro[f'{prefix}ATR_14'] = 0 # Valor placeholder ou NAN, dependendo da sua estratégia

        # Coletar nomes das features
        features_macro.extend([
            f'{prefix}Ret_21d', f'{prefix}Vol_21d', f'{prefix}vs_SMA50', 
            f'{prefix}RSI_14', f'{prefix}MACD_Hist'
        ])
        if has_ohl:
             features_macro.append(f'{prefix}ATR_14')

    # -----------------------------------------------
    ## ✅ Seleção e Saída
    # -----------------------------------------------
    
    # Dropna remove as linhas com NaN geradas pelas janelas dos indicadores
    return data_macro[features_macro].dropna()

Funcional

In [19]:
def baixar_dados_macro(start="2010-01-01", end=None):
    """
    Baixa dados de indicadores macro/globais (IBOV e Ouro) e calcula retornos.
    """
    # Mapeamento dos Tickers (IBOV e Ouro Future)
    macro_tickers = ['^BVSP', 'GC=F'] 
    
    if end is not None:
        data_macro = yf.download(macro_tickers, start=start, end=end, progress=False, auto_adjust=True)
    else:
        data_macro = yf.download(macro_tickers, start=start, progress=False, auto_adjust=True)
        
    if data_macro.empty:
        return None

    # Reduzindo para o preço de fechamento
    # O MultiIndex do yfinance é 'Close', 'Volume' e o Ticker
    data_close = data_macro['Close']
    
    # Renomeando as colunas
    data_close.columns = ['IBOV_Close', 'OURO_Close']
    
    # Cálculo dos Retornos Diários (O Retorno é a Feature)
    data_close['IBOV_Ret'] = data_close['IBOV_Close'].pct_change(21)
    data_close['OURO_Ret'] = data_close['OURO_Close'].pct_change(21)
    
    # Indicadores Adicionais de Volatilidade (Opcional)
    data_close['IBOV_Vol_21d'] = data_close['IBOV_Close'].pct_change().rolling(21).std()

    # Selecionar apenas as features que serão usadas no modelo
    features_macro = ['IBOV_Ret', 'OURO_Ret', 'IBOV_Vol_21d']
    
    # Dropna remove a primeira linha do Retorno
    return data_close[features_macro].dropna()

In [20]:
def unificar_dados_e_preparar(df_acao, df_macro):
    """
    Une os dados da ação e os dados macro, alinhando-os pelo índice de data.
    """
    
    # ⚠️ A união usando 'inner' (padrão) garante que apenas datas que existem 
    # tanto na AÇÃO quanto nos dados MACRO sejam mantidas.
    # Isso é essencial, pois o Ouro pode ter negociação em feriados diferentes do IBOV/Ação.
    
    df_final = pd.merge(
        df_acao, 
        df_macro, 
        left_index=True, 
        right_index=True, 
        how='inner' # Use 'inner' para remover datas incompletas/desalinhadas
    )
    
    # Limpeza final, se alguma variável macro gerou NA no merge, removemos a linha
    df_final = df_final.dropna()
    
    # Removendo colunas que não são features ou target (ex: Ticker, Close, Volume, etc.)
    # Mantenha apenas o que será alimentado no modelo.
    colunas_a_remover = [col for col in df_final.columns if col in ['Close', 'Open', 'High', 'Low', 'Volume', 'Ticker', 'Ret_Simples_21d_Passado']]
    
    df_modelo = df_final.drop(columns=colunas_a_remover, errors='ignore')
    
    return df_modelo

In [21]:
def treinar_modelo(
    df_acao: pd.DataFrame, 
    target_col: str, 
    model_type: str = 'regressor', # 'regressor' ou 'classifier'
    rf_params: Dict[str, Any] = None
) -> Union[RandomForestRegressor, RandomForestClassifier]:
    """
    Treina um modelo Random Forest (Regressor ou Classifier) para uma única ação.
    
    Args:
        df_acao: DataFrame processado de uma única ação (com features e target).
        target_col: Nome da coluna TARGET (ex: 'TARGET_Ret_21d' para Regressor).
        model_type: Tipo de modelo a ser treinado ('regressor' ou 'classifier').
        rf_params: Dicionário com hiperparâmetros para o Random Forest.
    
    Returns:
        Um modelo Random Forest treinado.
    """
    
    # 1. Definição de Hiperparâmetros Padrão
    # ATENÇÃO: max_depth e n_estimators são os mais importantes para evitar overfitting.
    default_params = {
        'n_estimators': 200,
        'max_depth': 12, # Limite de profundidade (regulador crucial)
        'min_samples_leaf': 5, # Exige 5 amostras em um nó para dividi-lo
        'random_state': 42,
        'n_jobs': -1 # Usa todos os núcleos da CPU para velocidade
    }
    
    if model_type == 'classifier':
        default_params['class_weight'] = 'balanced' # Necessário para targets desbalanceados
    
    params = default_params if rf_params is None else rf_params
    
    # 2. Separação de X e y
    features = [col for col in df_acao.columns if (col != target_col and col != 'TARGET')]
    
    X_train, y_train = df_acao[features], df_acao[target_col]
    
    # 3. Escolha do Modelo e Treinamento
    if model_type == 'regressor':
        model = RandomForestRegressor(**params)
        print("Treinando Random Forest Regressor...")
    elif model_type == 'classifier':
        model = RandomForestClassifier(**params)
        print("Treinando Random Forest Classifier...")
    else:
        raise ValueError("model_type deve ser 'regressor' ou 'classifier'.")
        
    model.fit(X_train, y_train)
    
    print(f"Modelo treinado com sucesso. Total de amostras: {len(X_train)}")
    
    return model

In [12]:
def limpar_e_winsorizar_dados(df: pd.DataFrame, target_col: str, limite_winsor: float = 0.01) -> pd.DataFrame:
    """
    Trata valores infinitos, NaNs e realiza a Winsorização 
    (limitação de outliers) no target e features chave.
    
    Args:
        df: DataFrame com features e target.
        target_col: Nome da coluna target (ex: 'TARGET_Ret_21d').
        limite_winsor: Percentual para winsorização (ex: 0.01 significa 1% e 99%).
        
    Returns:
        DataFrame limpo e tratado.
    """
    
    # 1. Tratar Infinitos e NaN (Essencial para o Random Forest)
    
    # a. Converter Infinitos para NaN em todo o DataFrame
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # b. Descartar linhas com qualquer NaN (devido aos indicadores, targets, ou infinitos)
    df.dropna(inplace=True)
    
    if df.empty:
        print("Atenção: DataFrame ficou vazio após limpeza de NaNs/Infinitos.")
        return df

    # 2. Winsorização do TARGET (Reduz Outliers Severos)
    # Identifica o 1º e o 99º percentil no TARGET
    limite_inferior = df[target_col].quantile(limite_winsor) # 1%
    limite_superior = df[target_col].quantile(1 - limite_winsor) # 99%
    
    # Aplica o limite: valores fora do intervalo [1%, 99%] são substituídos pelos limites.
    df[target_col].clip(lower=limite_inferior, upper=limite_superior, inplace=True)

    # 3. Winsorização de Features Chave (Opcional, mas recomendado para Retornos)
    
    # Aplicar a mesma lógica para o retorno diário e retornos lag
    for col in df.columns:
        if col.startswith('Ret_') and 'TARGET' not in col:
            q_low = df[col].quantile(limite_winsor)
            q_high = df[col].quantile(1 - limite_winsor)
            df[col].clip(lower=q_low, upper=q_high, inplace=True)
            
    print(f"Tratamento de Infinitos e Winsorização ({limite_winsor*100}%) concluído.")
    return df

In [13]:
def avaliar_r2_in_sample(modelo, df_acao: pd.DataFrame, features: list, target_col: str):
    """
    Calcula o R² nos dados de TREINO.
    """
    X_train = df_acao[features]
    y_train = df_acao[target_col]
    
    # Previsão nos dados de treino
    y_pred_train = modelo.predict(X_train)
    
    # Calcula o R²
    r2 = r2_score(y_train, y_pred_train)
    
    return r2

In [14]:
def calcular_beta_alfa_capm(retornos_ativo, retornos_mercado, taxa_livre_risco, window=63):
    """Calcula o Beta e o Alfa de Jensen (não anualizado) usando regressão rolling."""
    

    retornos_ativo_excesso = retornos_ativo - taxa_livre_risco
    retornos_mercado_excesso = retornos_mercado - taxa_livre_risco
    
    # 2. Função de Regressão Rolling
    def run_regression(series):
        # A série contém os últimos 'window' pares de (Retorno_Ativo, Retorno_Mercado)
        Y = series.iloc[:, 0]  # Retornos do Ativo em Excesso
        X = series.iloc[:, 1]  # Retornos do Mercado em Excesso
        
        # Adicionar uma constante para calcular o Alpha (Intercepto)
        X = sm.add_constant(X)
        
        try:
            model = sm.OLS(Y, X).fit()
            # Beta é o coeficiente da variável de mercado (índice 1)
            # Alpha é o intercepto (constante, índice 0)
            return pd.Series([model.params.iloc[1], model.params.iloc[0]], index=['Beta', 'Alpha'])
        except (ValueError, np.linalg.LinAlgError):
            # Lidar com janelas onde a regressão falha (ex: dados constantes)
            return pd.Series([np.nan, np.nan], index=['Beta', 'Alpha'])

    # 3. Aplicar a Regressão Rolling
    # Cria um DataFrame temporário com os dois retornos em excesso
    df_temp = pd.DataFrame({'Ativo': retornos_ativo_excesso, 
                            'Mercado': retornos_mercado_excesso})
    
    # Aplica a função de regressão a cada janela
    results = df_temp.rolling(window=window).apply(run_regression, raw=False)
    
    # Retorna o Beta e o Alpha como colunas separadas
    return results['Beta'], results['Alpha']

# --- EXEMPLO DE USO ---
# data['R_f'] = (0.05 / 252) # Exemplo: 5% anual, divididos por 252 dias úteis
# data['IBOV_Ret'] = data['IBOV_Close'].pct_change()

# data['Beta_IBOV_63d'], data['Alpha_IBOV_63d'] = calcular_beta_alfa_capm(
#     data['Ret_1d'], 
#     data['IBOV_Ret'], 
#     data['R_f'], 
#     window=63 # Janela de 3 meses de negociação
# )

In [16]:
def avaliar_modelo_in_sample():
    # Passo 1: Baixar e calcular MACRO
    df_macro = baixar_dados_macro(start="2010-01-01")

    # Passo 2: Baixar e calcular a Ação (usando a função melhorada com targets e lags)
    # (Assumindo que sua função baixou a ação WEGE3.SA)
    df_acao = baixar_e_calcular_indicadores(ticker="WEGE3.SA", start="2010-01-01")

    # Passo 3: Unificar os dados
    if df_acao is not None and df_macro is not None:
        df = unificar_dados_e_preparar(df_acao, df_macro)
    target = 'TARGET_Ret' # O retorno contínuo de 21 dias à frente
    tipo = 'regressor'
    # Hiperparâmetros Otimizados (Exemplo: Valores que você encontrou após um Grid Search)
    params_regressor = {
        'n_estimators': 289, 
        'max_depth': 23,
        'min_samples_leaf': 9,
        'max_features': 0.6  # Usar 60% das features em cada split
    }

    df_limpo = limpar_e_winsorizar_dados(
            df=df, 
            target_col='TARGET_Ret', 
            limite_winsor=0.005 # Exemplo: 0.5% e 99.5%
        )

    # Chamada da Função
    modelo_regressor = treinar_modelo(
        df_acao=df,
        target_col=target,
        model_type=tipo,
        rf_params=params_regressor
    )
    print(f"Modelo treinado: {type(modelo_regressor)}")

    # Supondo que modelo_regressor foi obtido da sua chamada a treinar_modelo

    # Acessa a lista de nomes das features diretamente do modelo treinado
    features_usadas = modelo_regressor.feature_names_in_
    # Supondo que modelo_regressor, features_usadas e df_limpo já foram definidos

    # 🎯 Passo A: Calcular o R² na Amostra (In-Sample)
    r2_na_amostra = avaliar_r2_in_sample(
        modelo=modelo_regressor,
        df_acao=df_limpo,
        features=features_usadas,
        target_col='TARGET_Ret'
    )

    print(f"### 📈 Desempenho In-Sample (R²) ")
    print(f"R² (Treino): {r2_na_amostra:.4f}")
    print("---")

    # 🎯 Passo B: Calcular a Importância das Features (Top 10)
    importancia = pd.Series(
        modelo_regressor.feature_importances_, 
        index=features_usadas
    ).sort_values(ascending=False)

    # Imprimindo as Top 10 features
    print("### 🥇 Top 10 Features Mais Importantes")
    print(importancia.head(50))
    print("---")

    # 🎯 Passo C: Configurações Chave
    print("### ⚙️ Configurações Chave do Modelo (Reguladores)")
    print(f"Profundidade Máxima (max_depth): {modelo_regressor.get_params().get('max_depth')}")
    print(f"Amostras Mínimas por Folha (min_samples_leaf): {modelo_regressor.get_params().get('min_samples_leaf')}")

    return modelo_regressor, df_limpo, features_usadas

In [22]:
modelo_regressor, df_limpo, features_usadas = avaliar_modelo_in_sample()

Tratamento de Infinitos e Winsorização (0.5%) concluído.
Treinando Random Forest Regressor...
Modelo treinado com sucesso. Total de amostras: 3811
Modelo treinado: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
### 📈 Desempenho In-Sample (R²) 
R² (Treino): 0.2552
---
### 🥇 Top 10 Features Mais Importantes
IBOV_Ret        0.104691
OURO_Ret        0.103380
MACD_Hist       0.102930
Vol_21d         0.100610
IBOV_Vol_21d    0.090258
SMA_25-30       0.090081
RSI_14          0.089732
SMA_21          0.080509
SMA_25-50       0.080359
SMA_15-20       0.079429
SMA_10-20       0.078020
dtype: float64
---
### ⚙️ Configurações Chave do Modelo (Reguladores)
Profundidade Máxima (max_depth): 23
Amostras Mínimas por Folha (min_samples_leaf): 9


In [23]:
X = df_limpo[features_usadas]
y = df_limpo['TARGET_Ret']

# 2. Definição do Espaço de Busca (Range de Valores)
# Use distribuições discretas (randint) para n_estimators, max_depth, etc.
param_dist = {
    'n_estimators': randint(low=200, high=250), # De 100 a 500 árvores
    'max_depth': randint(low=15, high=25),       # Profundidade entre 8 e 25 (para subajuste)
    'min_samples_leaf': randint(low=5, high=10),# Mínimo de amostras por folha
    'max_features': [0.6, 0.8, 1.0, 'sqrt', 'log2'] # Número de features a considerar
}

# 3. Inicialização do Modelo Base
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# 4. Configuração da Validação Cruzada Temporal
# 'n_splits=5' significa que o treino será dividido em 5 iterações temporais
# O treino será em 20% do tempo, depois 40%, 60%, 80%, e o teste no período seguinte.
tscv = TimeSeriesSplit(n_splits=5) 

# --- B. Execução do Randomized Search ---

# 'n_iter=50' significa que ele testará 50 combinações aleatórias
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=5, 
    scoring='r2', # Métrica a ser otimizada
    cv=tscv,      # Validação Cruzada de Séries Temporais
    verbose=2,    # Mostra o progresso
    random_state=42,
    n_jobs=-1
)

# Inicia a busca (Isso pode levar tempo!)
print("Iniciando otimização... Testando 50 combinações.")
random_search.fit(X, y)

# --- C. Resultados ---

# 1. Os melhores parâmetros encontrados
melhores_params = random_search.best_params_
print("\n### 🎉 Melhores Hiperparâmetros Encontrados:")
print(melhores_params)

# 2. O melhor score de R² médio
melhor_r2_medio = random_search.best_score_
print(f"\n### 🎯 Melhor R² Médio da Validação Cruzada: {melhor_r2_medio:.4f}")

# O melhor modelo para uso
modelo_otimizado = random_search.best_estimator_

Iniciando otimização... Testando 50 combinações.
Fitting 5 folds for each of 5 candidates, totalling 25 fits

### 🎉 Melhores Hiperparâmetros Encontrados:
{'max_depth': 21, 'max_features': 'sqrt', 'min_samples_leaf': 9, 'n_estimators': 214}

### 🎯 Melhor R² Médio da Validação Cruzada: -0.0339
